# Clipflow AI UGC Avatar — OmniAvatar 1.3B on Kaggle (100% free)

**Setup (do this in Kaggle UI before running):**
1. Settings → Accelerator → **GPU T4 x2** (or T4 x1)
2. Settings → Internet → **ON**
3. Edit the CONFIG cell below: paste your script, pick a voice, set an avatar photo URL (a front-facing human portrait)
4. Run All

Output: a 480p 9:16-ish talking-avatar clip with your script, lip-synced. Download it from `/kaggle/working/`.

**Model:** OmniAvatar 1.3B (Apache 2.0, audio-driven avatar on Wan2.1). **Voice:** edge-tts (free Microsoft Edge voices, no API key).

⚠️ First run downloads ~10GB of models and takes 20-40 min setup. Renders: ~5-15 min per clip of script.

In [ ]:
# ===================== CONFIG — EDIT THIS =====================
SCRIPT = """Stop scrolling. If you are editing short-form video every single day, the boring parts are eating your time. Clipflow renders clean seventy-two-oh-p vertical videos right on your device. Free forever."""

# Voice: try en-US-AvaNeural, en-US-AndrewNeural, en-GB-SoniaNeural, en-US-BrianNeural
VOICE = "en-US-AvaNeural"

# Front-facing portrait photo (direct .jpg/.png URL). Leave "" to use a built-in sample face.
AVATAR_URL = ""

# Population description for the avatar motion: [first frame] - [behavior] - [background]
PROMPT = "A young woman holding a smartphone, filming in selfie style - she talks directly to the camera with expressive hand gestures and slight head movements - bright modern kitchen background"

NUM_STEPS = 25          # 20-50, higher = better quality, slower
AUDIO_SCALE = 6         # lip-sync strength 4-6 low, raise to 7-8 if lips look off


In [ ]:
!git clone https://github.com/Omni-Avatar/OmniAvatar /kaggle/working/OmniAvatar
!pip install -q torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu124
!pip install -q "huggingface_hub[cli]" edge-tts
!pip install -q -r /kaggle/working/OmniAvatar/requirements.txt

In [ ]:
import os
os.chdir('/kaggle/working/OmniAvatar')

# Download 1.3B stack (~10GB, skipped if cached)
!huggingface-cli download Wan-AI/Wan2.1-T2V-1.3B --local-dir ./pretrained_models/Wan2.1-T2V-1.3B
!huggingface-cli download facebook/wav2vec2-base-960h --local-dir ./pretrained_models/wav2vec2-base-960h
!huggingface-cli download OmniAvatar/OmniAvatar-1.3B --local-dir ./pretrained_models/OmniAvatar-1.3B
print('models ready')

In [ ]:
import os, glob, asyncio
import edge_tts, subprocess
os.chdir('/kaggle/working/OmniAvatar')

# 1) TTS -> audio
async def tts():
    await edge_tts.Communicate(SCRIPT, VOICE).save('/kaggle/working/script_voice.mp3')
asyncio.run(tts())

# 2) convert to 16k mono wav
subprocess.run(['ffmpeg', '-y', '-i', '/kaggle/working/script_voice.mp3',
                '-ar', '16000', '-ac', '1', '/kaggle/working/script_voice.wav'], check=True)

# 3) avatar image: user URL or a portrait bundled in the repo examples
avatar = '/kaggle/working/avatar.png'
if AVATAR_URL:
    subprocess.run(['wget', '-q', '-O', avatar, AVATAR_URL], check=True)
else:
    cands = sorted(glob.glob('examples/**/*.png', recursive=True) + glob.glob('examples/**/*.jpg', recursive=True))
    assert cands, 'No example image found — set AVATAR_URL to a portrait photo URL'
    subprocess.run(['cp', cands[0], avatar], check=True)

# 4) write the batch file: prompt@@image@@audio
with open('/kaggle/working/infer_samples.txt', 'w') as f:
    f.write(f'{PROMPT}@@{avatar}@@/kaggle/working/script_voice.wav')
print('audio + avatar + sample file ready')

In [ ]:
import os, subprocess
os.chdir('/kaggle/working/OmniAvatar')

cmd = ['torchrun', '--standalone', '--nproc_per_node=1', 'scripts/inference.py',
       '--config', 'configs/inference_1.3B.yaml',
       '--input_file', '/kaggle/working/infer_samples.txt',
       '--hp', f'num_steps={NUM_STEPS},guidance_scale=4.5,audio_scale={AUDIO_SCALE},overlap_frame=13,tea_cache_l1_thresh=0.14,num_persistent_param_in_dit=0']
print(' '.join(cmd))
subprocess.run(cmd)
subprocess.run(['find', '.', '-name', '*.mp4', '-mmin', '-90', '-exec', 'ls', '-lh', '{}', ';'])

In [ ]:
import shutil, glob, os
# copy fresh outputs to /kaggle/working so they show in the Output panel
os.chdir('/kaggle/working/OmniAvatar')
vids = [p for p in glob.glob('**/*.mp4', recursive=True)]
for i, v in enumerate(vids):
    dst = f'/kaggle/working/ugc_clip_{i}.mp4'
    shutil.copy(v, dst)
    print('saved:', dst)